In [1]:
!pip install transformers datasets torch scikit-learn -q

import torch
from datasets import load_dataset
from transformers import (DistilBertTokenizer,
                          DistilBertForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU name: Tesla T4


In [5]:
# Load dataset
dataset = load_dataset("deepset/prompt-injections")

# Split the 'train' split into 'train' and 'validation' (10% for validation)
split_dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# Re-organize into a new DatasetDict
from datasets import DatasetDict
dataset = DatasetDict({
    'train': split_dataset['train'],
    'validation': split_dataset['test'], # This is our new validation set
    'test': dataset['test']
})

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess(examples):
    text = [t.lower().strip() for t in examples["text"]]
    return tokenizer(text, truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(preprocess, batched=True)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
print("New Splits:", tokenized)

Map:   0%|          | 0/491 [00:00<?, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

New Splits: DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 491
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 55
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 116
    })
})


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./shieldprompt_checkpoints",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Uses the GPU more efficiently
)

# ... (Keep your compute_metrics and model code the same) ...

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized["train"],
    eval_dataset    = tokenized["validation"], # This will now find the key!
    compute_metrics = compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.237907,0.927273,0.882353,1.000000,0.789474
2,No log,0.122503,0.963636,0.947368,0.947368,0.947368
3,No log,0.143642,0.963636,0.944444,1.000000,0.894737
4,No log,0.113225,0.981818,0.972973,1.000000,0.947368


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=124, training_loss=0.1440030836289929, metrics={'train_runtime': 35.1384, 'train_samples_per_second': 55.893, 'train_steps_per_second': 3.529, 'total_flos': 65041492740096.0, 'train_loss': 0.1440030836289929, 'epoch': 4.0})

In [7]:
from google.colab import files
import shutil

# Save and Zip
trainer.save_model("./shieldprompt_distilbert")
tokenizer.save_pretrained("./shieldprompt_distilbert")
shutil.make_archive("shieldprompt_distilbert", "zip", "./shieldprompt_distilbert")

# Download
files.download("shieldprompt_distilbert.zip")
files.download("confusion_matrix.png")
files.download("roc_curve.png")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: confusion_matrix.png